In [26]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn import preprocessing
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Import data


In [2]:
# read data
df = pd.read_csv('2018-06-06-ss.cleaned.csv')

In [3]:

df.shape

(393732, 7)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 393732 entries, 0 to 393731
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   pdb_id         393732 non-null  object
 1   chain_code     393732 non-null  object
 2   seq            393732 non-null  object
 3   sst8           393732 non-null  object
 4   sst3           393732 non-null  object
 5   len            393732 non-null  int64 
 6   has_nonstd_aa  393732 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 18.4+ MB


Description of columns

1. pdb_id: the id used to locate its entry on https://www.rcsb.org/
2. chain_code: when a protein consists of multiple peptides (chains), the chain code is needed to locate a particular one.
3. seq: the sequence of the peptide
4. sst8: the eight-state (Q8) secondary structure
5. sst3: the three-state (Q3) secondary structure
6. len: the length of the peptide
7. has_nonstd_aa: whether the peptide contains nonstandard amino acids (B, O, U, X, or Z).

1. Preprocessing data

In [5]:
df = df[df['has_nonstd_aa'] == False]

In [6]:
#Encoding
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_int = {aa: i for i, aa in enumerate(amino_acids)}

def encode_sequence(seq):
    return [aa_to_int[aa] for aa in seq]

df['encoded_seq'] = df['seq'].apply(encode_sequence)

In [15]:
max_seq_len = df['len'].max()  # or use a fixed value
# Convert sequences to PyTorch tensors
encoded_tensors = [torch.tensor(seq) for seq in df['encoded_seq']]

In [36]:
# Pad sequences
padded_seq = nn.utils.rnn.pad_sequence(encoded_tensors, batch_first=True)

In [39]:
# Encoding sst8
label_encoder_8 = preprocessing.LabelEncoder()
df['encoded_sst8'] = label_encoder_8.fit_transform(df['sst8'])

In [38]:
labels = torch.tensor(df['encoded_sst8'].values)
#labels

In [40]:
# Create a TensorDataset from the padded sequences and labels
dataset = TensorDataset(padded_seq, labels)

padded_seq_train, padded_seq_test, labels_train, labels_test = train_test_split(
    padded_seq, labels, test_size=0.2, random_state=42)


train_dataset = TensorDataset(padded_seq_train, labels_train)
test_dataset = TensorDataset(padded_seq_test, labels_test)

In [28]:
# Create DataLoaders for training and testing
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

CNN Model

In [46]:
class CNN(nn.Module):
    def __init__(self, num_classes=8):  # Now predicting 8 classes for sst8
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, (5, 1), padding=(2, 0))
        self.p1 = nn.MaxPool2d((2, 1), (2, 1))
        self.conv2 = nn.Conv2d(32, 64, (5, 1), padding=(2, 0))
        self.p2 = nn.MaxPool2d((2, 1), (2, 1))
        self.flatten = nn.Flatten()
        self.d1 = nn.Linear(64 * (padded_seq.shape[1] // 4), 1024)  # Adjust for pooling
        self.dropout = nn.Dropout(0.2)
        self.d2 = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = x.float()
        x = x.squeeze()
        x = x.unsqueeze(1).unsqueeze(3)  # Add channel dimensions
        conv1 = torch.relu(self.conv1(x))
        p1 = self.p1(conv1)
        conv2 = torch.relu(self.conv2(p1))
        p2 = self.p2(conv2)
        flatten = self.flatten(p2)
        d1 = torch.relu(self.d1(flatten))
        dropout = self.dropout(d1)
        out = torch.softmax(self.d2(dropout), dim=1)
        return conv1, p1, conv2, p2, out
    
model = CNN()

In [19]:
# Loss func and Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

loss_object = nn.CrossEntropyLoss()

In [20]:
# Train and test steps
def train_step(images, labels):
    optimizer.zero_grad()  # Zero the gradients
    _, _, _, _, predictions = model(images)  # Forward pass
    loss = loss_object(predictions, labels)  # Compute the loss
    loss.backward()  # Backpropagation
    optimizer.step()  # Update weights
    return loss.item(), (predictions.argmax(dim=1) == labels).float().mean().item()

def test_step(images, labels):
    _, _, _, _, predictions = model(images)  # Forward pass (no gradients)
    t_loss = loss_object(predictions, labels)  # Compute test loss
    return t_loss.item(), (predictions.argmax(dim=1) == labels).float().mean().item()

In [47]:
num_epochs = 100
test_acc = []

for epoch in range(num_epochs):
    train_loss = 0.0
    train_accuracy = 0.0
    test_loss = 0.0
    test_accuracy = 0.0
    train_steps = 0
    test_steps = 0

    # Training loop
    for images, labels in train_dataloader:
        images = images.unsqueeze(1).unsqueeze(3)  # Add dimensions for CNN input
        loss, accuracy = train_step(images, labels)
        train_loss += loss
        train_accuracy += accuracy
        train_steps += 1

    # Testing loop (no gradient calculations)
    with torch.no_grad():
        for test_images, test_labels in test_dataloader:
            test_images = test_images.unsqueeze(1).unsqueeze(3)  # Add dimensions for CNN input
            t_loss, t_accuracy = test_step(test_images, test_labels)
            test_loss += t_loss
            test_accuracy += t_accuracy
            test_steps += 1

    test_acc.append(test_accuracy / test_steps)  # Save test accuracy for this epoch
    
    # Print results
    template = 'Epoch {}, Loss: {:.4f}, Accuracy: {:.4f}, Test Loss: {:.4f}, Test Accuracy: {:.4f}'
    print(template.format(epoch + 1,
                          train_loss / train_steps,
                          train_accuracy / train_steps,
                          test_loss / test_steps,
                          test_accuracy / test_steps))

IndexError: Target 176903 is out of bounds.